# Clase 115 — Learning rate scheduling

Variar el LR durante el entrenamiento porque ningún LR es óptimo en todas las fases. Estrategias: step, exponential, **cosine annealing** (default moderno) y **warmup + decay** (estándar en Transformers).

Requiere: `tensorflow` / `keras` (≥ 3.0), `numpy`, `matplotlib`.

## 1. `ExponentialDecay`: `lr = lr_0 · γ^(step/decay_steps)`

Un schedule se pasa directamente como `learning_rate=` del optimizer.

In [ ]:
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers
keras.utils.set_random_seed(42)

exp = keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate=1e-3, decay_steps=1000, decay_rate=0.9)
print("LR exponencial en steps 0, 1000, 5000:",
      [round(float(exp(s)), 6) for s in (0, 1000, 5000)])
opt = keras.optimizers.Adam(learning_rate=exp)      # el schedule vive dentro del optimizer

## 2. `PiecewiseConstantDecay`: escalones por tramos

Útil para el clásico "cortá el LR a la mitad en las épocas N y M".

In [ ]:
pw = keras.optimizers.schedules.PiecewiseConstantDecay(
    boundaries=[1000, 3000], values=[1e-3, 5e-4, 1e-4])
print("piecewise en steps 500 / 2000 / 4000:",
      [float(pw(s)) for s in (500, 2000, 4000)])

## 3. `CosineDecay` con warmup

Keras 3 soporta warmup nativo: sube linealmente de 0 a `warmup_target` en `warmup_steps` y luego baja con coseno.

In [ ]:
cos = keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=0.0, warmup_target=1e-3, warmup_steps=500,
    decay_steps=10_000, alpha=0.0)
lrs = [round(float(cos(s)), 6) for s in (0, 250, 500, 5000, 10_000)]
print("cosine + warmup (sube hasta 1e-3 y luego decae):", lrs)

## 4. `LearningRateScheduler`: función época → LR

Callback que ajusta un LR **escalar** en función de la época (por ejemplo, step decay).

In [ ]:
def step_decay(epoca, lr):
    return lr * 0.5 if (epoca > 0 and epoca % 10 == 0) else lr

lr_callback = keras.callbacks.LearningRateScheduler(step_decay, verbose=0)
print("LearningRateScheduler aplica una función (época, lr) -> nuevo lr")

## 5. `ReduceLROnPlateau`: reactivo

Baja el LR cuando una métrica deja de mejorar (a diferencia del schedule, que es proactivo).

In [ ]:
reduce = keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6)
print("ReduceLROnPlateau baja el LR cuando val_loss se estanca durante 3 épocas")

## 6. Entrenar con cosine + warmup y extraer la curva de LR

In [ ]:
def mlp(learning_rate):
    m = keras.Sequential([
        keras.Input(shape=(784,)),
        layers.Dense(128, activation="relu", kernel_initializer="he_normal"),
        layers.Dense(10,  activation="softmax"),
    ])
    m.compile(optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m

modelo = mlp(cos)
curva = [float(cos(s)) for s in range(0, 10_000, 500)]
print("modelo con cosine+warmup listo; puntos de la curva LR:", len(curva))
# import matplotlib.pyplot as plt; plt.plot(range(0, 10_000, 500), curva)

## Ejercicios

1. **Schedule básica**: `CosineDecay(...)` dentro de `Adam` y entrenar; graficá `val_loss`.
2. **Visualizar el LR**: evaluá una schedule en varios steps y graficá la curva.
3. **Warmup + Cosine**: compará contra sin warmup en un modelo chico.
4. **ReduceLROnPlateau vs Cosine**: compará el reactivo contra el proactivo.

## Conclusiones

- Un LR fijo arranca bien pero termina demasiado alto para refinar.
- Los schedules (`ExponentialDecay`, `PiecewiseConstantDecay`, `CosineDecay`) se pasan como `learning_rate=` del optimizer.
- **Cosine con warmup** es el estándar moderno (visión y NLP).
- `LearningRateScheduler` (proactivo, por época) y `ReduceLROnPlateau` (reactivo) son alternativas basadas en callbacks; no combinar un schedule del optimizer con estos callbacks.
- Calibrar `decay_steps = epochs · steps_per_epoch` para que no decaiga demasiado rápido.

## ✅ Soluciones de los ejercicios

Learning rate scheduling en Keras 3 / tf.keras sobre Fashion-MNIST. Con `tensorflow` no instalado las celdas se validan por AST; con TF instalado corren tal cual. Las estrategias cubren los 5 ejercicios: cosine, visualización de la curva, warmup+cosine, ReduceLROnPlateau y one-cycle.

**Ej. 1 — Schedule básica.** `CosineDecay` pasada como `learning_rate=` al `Adam`; entrenar y mirar `val_loss`.

In [ ]:
import numpy as np
from tensorflow import keras

(x_train, y_train), (x_val, y_val) = keras.datasets.fashion_mnist.load_data()
x_train, x_val = x_train / 255., x_val / 255.

lr = keras.optimizers.schedules.CosineDecay(initial_learning_rate=1e-3, decay_steps=10_000)
model = keras.Sequential([keras.Input((28, 28)), keras.layers.Flatten(),
                          keras.layers.Dense(128, activation="relu"),
                          keras.layers.Dense(10, activation="softmax")])
model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
hist = model.fit(x_train, y_train, validation_data=(x_val, y_val),
                 epochs=5, batch_size=32, verbose=0)
print("val_loss final:", hist.history["val_loss"][-1])
# plt.plot(hist.history["val_loss"]); plt.xlabel("epoch"); plt.ylabel("val_loss")

**Ej. 2 — Visualizar el LR.** Evaluar la schedule en varios steps (es *callable*) y graficar la curva.

In [ ]:
from tensorflow import keras

sched = keras.optimizers.schedules.CosineDecay(1e-3, decay_steps=10_000)
for step in [0, 100, 1000, 5000, 10000]:
    print(f"step {step:6d} -> lr {float(sched(step)):.6f}")
# curva completa:
# import numpy as np
# steps = np.arange(0, 10_000)
# plt.plot(steps, [float(sched(s)) for s in steps]); plt.xlabel("step"); plt.ylabel("lr")

**Ej. 3 — Warmup + Cosine.** Keras 3 soporta warmup nativo: sube 0->`warmup_target` en `warmup_steps` y luego cosine.

In [ ]:
from tensorflow import keras

cos_warm = keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=0.0, decay_steps=9_000,
    warmup_target=1e-3, warmup_steps=1_000)   # 0 -> 1e-3 lineal, luego coseno
for step in [0, 500, 1000, 2000, 10000]:
    print(f"step {step:6d} -> lr {float(cos_warm(step)):.6f}")
# sin warmup: CosineDecay(1e-3, 10_000). En transformers, sin warmup los primeros
# gradientes (ruidosos) hacen diverger; con warmup el arranque es estable.

**Ej. 4 — ReduceLROnPlateau.** Alternativa **reactiva**: baja el LR cuando `val_loss` deja de mejorar.

In [ ]:
from tensorflow import keras

reduce = keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6)
# El LR arranca fijo en el optimizer; el callback lo baja x0.5 tras 3 epochs sin mejora:
# model.compile(optimizer=keras.optimizers.Adam(1e-3), loss="...", metrics=["accuracy"])
# model.fit(..., callbacks=[reduce])
# Reactivo (espera a estancarse) vs cosine, que es proactivo (baja siempre segun plan).
print("ReduceLROnPlateau(factor=0.5, patience=3): baja LR cuando val_loss se estanca")

**Ej. 5 — One-cycle.** Implementado con `LearningRateScheduler`: sube hasta `lr_max` en la 1a mitad y baja ~100x en la 2a.

In [ ]:
from tensorflow import keras

def one_cycle(total_epochs, lr_max):
    half = total_epochs / 2
    def sched(epoch, lr):
        if epoch < half:                       # rampa de subida
            return lr_max * (epoch + 1) / half
        down = (epoch - half) / half           # descenso + cola
        return lr_max * (1 - 0.99 * down)      # termina ~100x por debajo de lr_max
    return keras.callbacks.LearningRateScheduler(sched)

cb = one_cycle(total_epochs=20, lr_max=1e-2)
# model.fit(..., callbacks=[cb])  -> curva triangular de LR (Smith 2018, super-convergence)
print("One-cycle listo: warmup lineal + descenso con cola larga")